# Classification model: food insecurity risk

This notebook uses the synthetic household food and housing insecurity dataset to build a classification model. The example target is `food_insecure_label`, where 1 means the synthetic household is food insecure and 0 means it is not.

The model is written as an early-warning example. Columns that directly reveal the target, such as `food_security_status`, `meals_skipped_last_30d`, and `months_food_shortage_last_year`, are excluded to reduce leakage.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_PATH = Path("synthetic_food_housing_insecurity_100k.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("/mnt/data/synthetic_food_housing_insecurity_100k.csv")

assert DATA_PATH.exists(), f"Dataset not found: {DATA_PATH}"
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

## 1. Select target and remove leakage columns

The dataset includes several outcome columns. A predictive model should not train on fields that would not be known at prediction time or that encode the answer directly.

In [ ]:
TARGET = "food_insecure_label"

LEAKAGE_COLUMNS = [
    "household_id",
    "train_test_split",
    "food_insecure_label",
    "food_security_status",
    "overall_hardship_score",
    "policy_priority_segment",
    "meals_skipped_last_30d",
    "months_food_shortage_last_year",
    "pantry_use_last_30d",
    "housing_insecure_label",
]

features = [c for c in df.columns if c not in LEAKAGE_COLUMNS]
train_mask = df["train_test_split"].eq("train")

X_train = df.loc[train_mask, features]
y_train = df.loc[train_mask, TARGET]
X_test = df.loc[~train_mask, features]
y_test = df.loc[~train_mask, TARGET]

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])
print("Feature count:", len(features))
print("Target rate in train:", round(y_train.mean(), 3))
print("Target rate in test:", round(y_test.mean(), 3))

## 2. Build a preprocessing and modeling pipeline

Numeric columns are median-imputed and standardized. Categorical columns are imputed with the most frequent value and one-hot encoded. Logistic regression is used because its coefficients are easier to inspect than many black-box models.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
)


def make_one_hot_encoder():
    """Keep the notebook compatible with old and new scikit-learn versions."""
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)


categorical_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_cols = [c for c in features if c not in categorical_cols]

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), numeric_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", make_one_hot_encoder()),
        ]), categorical_cols),
    ]
)

model = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("model", LogisticRegression(
            max_iter=500,
            solver="saga",
            class_weight="balanced",
            random_state=42,
        )),
    ]
)

model.fit(X_train, y_train)
print("Model trained.")

## 3. Evaluate classification performance

The threshold below is 0.50. In a real case-management setting, the threshold should be chosen according to capacity and the cost of false negatives versus false positives.

In [ ]:
y_score = model.predict_proba(X_test)[:, 1]
y_pred = (y_score >= 0.50).astype(int)

metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred),
    "recall": recall_score(y_test, y_pred),
    "f1": f1_score(y_test, y_pred),
    "roc_auc": roc_auc_score(y_test, y_score),
    "average_precision": average_precision_score(y_test, y_score),
}

pd.Series(metrics).round(3)

In [ ]:
print(classification_report(y_test, y_pred, digits=3))
cm = pd.DataFrame(
    confusion_matrix(y_test, y_pred),
    index=["actual_0", "actual_1"],
    columns=["pred_0", "pred_1"],
)
cm

## 4. Tune threshold for service capacity

This example ranks households by predicted risk and shows how many true food-insecure households are captured in the highest-risk groups. This is often more useful than a single 0.50 cutoff.

In [ ]:
ranked = pd.DataFrame({
    "household_id": df.loc[~train_mask, "household_id"].values,
    "actual_food_insecure": y_test.values,
    "predicted_risk": y_score,
}).sort_values("predicted_risk", ascending=False)

for pct in [0.05, 0.10, 0.20, 0.30]:
    n = int(len(ranked) * pct)
    top = ranked.head(n)
    print(
        f"Top {int(pct*100):>2}%: households={n:,}, "
        f"precision={top['actual_food_insecure'].mean():.3f}, "
        f"share of all positives captured={top['actual_food_insecure'].sum() / y_test.sum():.3f}"
    )

## 5. Inspect model drivers

Positive coefficients increase predicted food insecurity risk. Negative coefficients reduce it. Coefficients are not causal effects; they only describe how this fitted model uses the synthetic features.

In [ ]:
feature_names = model.named_steps["preprocess"].get_feature_names_out()
coefs = model.named_steps["model"].coef_[0]
coef_table = (
    pd.DataFrame({"feature": feature_names, "coefficient": coefs})
    .assign(abs_coefficient=lambda d: d["coefficient"].abs())
    .sort_values("abs_coefficient", ascending=False)
)

coef_table.head(20)

In [ ]:
import matplotlib.pyplot as plt

top = coef_table.head(15).sort_values("coefficient")
plt.figure(figsize=(9, 6))
plt.barh(top["feature"], top["coefficient"])
plt.axvline(0, linewidth=1)
plt.title("Top logistic regression coefficients")
plt.xlabel("Coefficient")
plt.tight_layout()
plt.show()

## 6. Save a prediction sample

The output file can be used in a dashboard, fairness review, or policy-priority workflow.

In [ ]:
output = ranked.head(5000).copy()
output_path = Path("food_insecurity_classification_predictions_sample.csv")
output.to_csv(output_path, index=False)
output_path